<a href="https://colab.research.google.com/github/bokofod/RothC_SOC/blob/main/Eligible_Land_Analysis_(Bulk_Export_to_GDrive_MNG_Loop_at_20m_2m_18Mar26_1330).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eligible Land Analysis
**Earthbanc** | Google Earth Engine via Python API

Runs a multi-criteria eligible land analysis on any input geometry and exports two GeoJSON layers to Google Drive:
- ✅ **Eligible** — areas meeting all enabled criteria
- ❌ **Non-eligible** — areas within the AOI that fail one or more criteria

---
### How to use
1. Run **Cell 1** to install and authenticate (once per session)
2. Edit **Cell 2** — set your asset path and adjust any parameters
3. Run **Cells 3–7** in order
4. Export tasks run automatically — files appear in your Google Drive

In [ ]:
# ============================================================
# CELL 1 — Install dependencies and authenticate
# Run once per Colab session. You will be prompted to log in
# with your Google account that has GEE access.
# ============================================================

!pip install earthengine-api --quiet

import ee

# Authenticate — opens a browser tab for Google login
ee.Authenticate()

# Initialise with your GEE cloud project
# Replace with your own project ID if needed
ee.Initialize(project='plus-earthbanc-io')

print('✓ Earth Engine initialised')

✓ Earth Engine initialised


In [ ]:
# ============================================================
# CELL 2 — Configuration
# Edit ASSET_PATH and CONTROL_PANEL here. All other cells
# read from these variables — do not edit below this cell.
# ============================================================

# --- Geometry input ---
# Option A: your own GEE asset (FeatureCollection)
ASSET_PATH = 'projects/plus-earthbanc-io/assets/YOUR_ASSET_HERE'
USE_ASSET  = False

GAUL_COUNTRY = 'Mongolia'   # one task per aimag (FAO GAUL ADM1, n=21)

CONTROL_PANEL = {
    'forestLoss': {
        'startYear': 2014,
        'endYear':   2024,
        'enabled':   True,
    },
    'canopyHeight': {
        'threshold':  2,        # m — above this is excluded
        'useAsScope': False,
        'enabled':    True,
    },
    # Precipitation: REMOVED from the eligibility stack for Mongolia
    # (Mongolia is too arid for the default 400 mm threshold —
    # almost no land would pass. Excluded outright, not just toggled.)
    'slope': {
        'maxThreshold': 60,     # %
        'enabled':      True,
    },
    'lulc': {
        'enabled': True,
    },
    'waterways': {
        'enabled': True,        # → JRC GSW global fallback
    },
    'roads': {
        'enabled': True,        # → GRIP4/Asia global fallback
    },
    'buildings': {
        'enabled':       False,
        'minConfidence': 0.65,
        'bufferRules':   [(25, 3), (100, 10), (500, 20), (1e9, 30)],
    },
    'minPatchSize': {
        'enabled':   False,
        'minAreaHa': 0.5,
    },
    'analysis': {
        'vectorScale':       20,
        'numVectorChunks':   4,
        'useSimplification': False,
        'simplifyTolerance': 20,
        'maxPixels':         1e13,
    },
}

PROCESSING_SCALE = 'coarse'   # 20m per-aimag is the safe default for Mongolia
FINE_SCALE_M     = 2
COARSE_SCALE_M   = 20

DRIVE_FOLDER = 'EligibleLandExports_Mongolia'

print('✓ Configuration set')
print(f'  Country        : {GAUL_COUNTRY} (one task per aimag)')
print(f'  Export scale   : {COARSE_SCALE_M} m')
print(f'  Drive folder   : {DRIVE_FOLDER}')

✓ Configuration set
  Country        : Mongolia (one task per aimag)
  Export scale   : 20 m
  Drive folder   : EligibleLandExports_Mongolia


In [ ]:
# ============================================================
# CELL 3 — geometry setup
#
# Loads country boundary from FAO GAUL level0.
# Splits into district grid for named exports
#
# ============================================================

import math

mongolia_fc = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
      .filter(ee.Filter.eq('ADM0_NAME', 'Mongolia'))
)
mongolia_geom    = mongolia_fc.geometry()
mongolia_area_ha = mongolia_geom.area(maxError=1).getInfo() / 10_000
print(f'✓ Mongolia loaded | area: {mongolia_area_ha:,.0f} ha')

# Aimags = FAO GAUL level1 (21 provinces). One Export task per aimag.
# To switch to soum-level (~330 tasks), change 'level1' → 'level2'
# and 'ADM1_NAME' → 'ADM2_NAME' below.
aimag_fc = (
    ee.FeatureCollection('FAO/GAUL/2015/level1')
      .filter(ee.Filter.eq('ADM0_NAME', 'Mongolia'))
)
all_aimag_names = sorted([
    f['properties']['ADM1_NAME']
    for f in aimag_fc.getInfo()['features']
])

# Start the loop from this letter onwards (inclusive, case-insensitive).
# Set to '' to process every aimag.
START_FROM = 'O'

aimag_names = [n for n in all_aimag_names if n.upper() >= START_FROM.upper()]

print(f'✓ Found {len(all_aimag_names)} aimags total | processing {len(aimag_names)} from "{START_FROM}" onwards:')
for n in aimag_names:
    print(f'    • {n}')

print(f'\n  Processing scale : {COARSE_SCALE_M} m')
print(f'  Export folder    : {DRIVE_FOLDER}')

✓ Mongolia loaded | area: 156,043,556 ha
✓ Found 22 aimags total | processing 12 from "O" onwards:
    • O'mnogovi
    • O'vorxangai
    • Orxon
    • Selenge
    • Su'xbaatar
    • To'v
    • Ulaanbaatar
    • Uvs
    • Xentii
    • Xo'vsgol
    • Xovd
    • Zavxan

  Processing scale : 20 m
  Export folder    : EligibleLandExports_Mongolia


In [ ]:
# ============================================================
# CELL 4 — eligible land export at 20m
#
# Single raster covering all of Denmark at 20m resolution.
# Named: LOCATION_eligible_20m.tif
# Exported to DRIVE_FOLDER (EligibleLandExports_Denmark)
# ============================================================

import math, time, unicodedata, re

if globals().get('submitted_tasks_mn'):
    raise RuntimeError(
        f'{len(submitted_tasks_mn)} tasks already submitted in this session — '
        f'clear `submitted_tasks_mn = []` manually if you really want to resubmit.'
    )

def sanitise(name, max_len=64):
    """ASCII-only, filename-safe, length-capped.

    - Drops apostrophe-like characters outright so "O'mnogovi" → "Omnogovi"
      (rather than "O_mnogovi"). Cleaner, and consistent across U+0027,
      U+2018/2019 (curly quotes), and U+02BB/02BC (modifier apostrophes).
    - Strips diacritics, forces ASCII.
    - Collapses any other non-safe char to a single underscore.
    - Caps length so the resulting GEE task description stays under 100.
    """
    cleaned    = re.sub(r"[\u2018\u2019\u02bb\u02bc']", '', name)
    nfkd       = unicodedata.normalize('NFKD', cleaned)
    ascii_name = nfkd.encode('ASCII', 'ignore').decode('ASCII')
    safe       = re.sub(r'[^A-Za-z0-9_-]+', '_', ascii_name).strip('_') or 'unnamed'
    return safe[:max_len]

submitted_tasks_mn = []
cp = CONTROL_PANEL

def resample_continuous(image):
    return image.resample('bilinear')

def resample_categorical(image):
    return image

print(f'Submitting one task per aimag → {len(aimag_names)} tasks total\n')

# GRIP4 has no single 'Asia' asset — sat-io splits it into 7 regions.
# Mongolia sits in Middle-East-Central-Asia, but we merge all 7 (matching the
# JS app at GEE-ARR-land-Eligibility.js:2810) so the same code works anywhere.
# Built once and reused per aimag via filterBounds.
GRIP_REGIONS = [
    'Africa',
    'Central-South-America',
    'Europe',
    'North-America',
    'Oceania',
    'South-East-Asia',
    'Middle-East-Central-Asia',
]
grip_global = ee.FeatureCollection(
    [ee.FeatureCollection(f'projects/sat-io/open-datasets/GRIP4/{r}') for r in GRIP_REGIONS]
).flatten()

for aimag in aimag_names:
    try:
        safe = sanitise(aimag)
        desc = f'Mongolia_{safe}_eligible_{COARSE_SCALE_M}m'
        desc = desc[:100]   # ← new line, hard cap for GEE's task-description limit

        aoi_fc            = aimag_fc.filter(ee.Filter.eq('ADM1_NAME', aimag))
        aoi_geom          = aoi_fc.geometry()
        analysis_geometry = aoi_geom.buffer(50)

        # Forest loss
        hansen = ee.Image('UMD/hansen/global_forest_change_2025_v1_13').clip(analysis_geometry)
        loss_year = hansen.select('lossyear')
        had_loss  = (loss_year.gte(cp['forestLoss']['startYear'] - 2000)
                              .And(loss_year.lte(cp['forestLoss']['endYear'] - 2000))
                              .unmask(0, False))
        no_loss   = had_loss.Not().clip(analysis_geometry)

        # Canopy height
        canopy_height = ee.ImageCollection(
            'projects/meta-forest-monitoring-okw37/assets/CanopyHeight'
        ).mosaic().clip(analysis_geometry)
        veg_above   = canopy_height.gte(cp['canopyHeight']['threshold']).unmask(0, False).clip(analysis_geometry)
        no_loss_veg = no_loss.And(veg_above.Not()).byte().clip(analysis_geometry)

        # Slope (precipitation removed entirely)
        dem        = ee.Image('NASA/NASADEM_HGT/001').select('elevation').resample('bilinear').clip(analysis_geometry)
        slope_pct  = ee.Terrain.slope(dem).divide(180).multiply(math.pi).tan().multiply(100)
        slope_mask = slope_pct.lte(cp['slope']['maxThreshold']).unmask(0, False)

        all_criteria = no_loss_veg.And(slope_mask).byte()

        # LULC — ESA WorldCover plantable remap (matches JS app + Denmark notebook)
        esa = resample_categorical(
            ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(analysis_geometry)
        )
        plantable_mask = esa.remap(
            [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
            [ 1,  1,  1,  1,  0,  1,  0,  0,  0,  0,   0],
            defaultValue=0,
        )
        final_criteria = all_criteria.And(plantable_mask.eq(1)).byte().clip(aoi_geom)

        # Waterways — JRC Global Surface Water
        gsw        = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi_geom)
        water_mask = resample_continuous(gsw).gte(10).selfMask()
        water_excl = water_mask.focal_max(radius=20, units='meters', kernelType='circle').clip(aoi_geom)
        final_criteria = final_criteria.updateMask(water_excl.eq(0))

        # Roads — GRIP4 Asia (Mongolia is in the Asia partition, NOT Europe)
        grip_fc   = grip_global.filterBounds(aoi_geom)
        road_mask = ee.Image().byte().paint(featureCollection=grip_fc, color=1, width=1).clip(aoi_geom).selfMask()
        road_excl = road_mask.focal_max(radius=20, units='meters', kernelType='circle').clip(aoi_geom)
        final_criteria = final_criteria.updateMask(road_excl.eq(0))

        task = ee.batch.Export.image.toDrive(
            image          = final_criteria.unmask(0).byte(),
            description    = desc,
            folder         = DRIVE_FOLDER,
            fileNamePrefix = desc,
            scale          = COARSE_SCALE_M,
            crs            = 'EPSG:4326',
            region         = aoi_geom,
            maxPixels      = 1e13,
            fileFormat     = 'GeoTIFF',
        )
        task.start()
        submitted_tasks_mn.append(task)
        print(f'  ✓ {desc}')
        time.sleep(1)

    except Exception as e:
        print(f'  ✗ SKIPPED {aimag}: {e}')

print(f'\n✓ {len(submitted_tasks_mn)} tasks submitted to Drive folder: "{DRIVE_FOLDER}"')
print('Run Cell 5 to monitor.')

Submitting one task per aimag → 12 tasks total

  ✓ Mongolia_Omnogovi_eligible_20m
  ✓ Mongolia_Ovorxangai_eligible_20m
  ✓ Mongolia_Orxon_eligible_20m
  ✓ Mongolia_Selenge_eligible_20m
  ✓ Mongolia_Suxbaatar_eligible_20m
  ✓ Mongolia_Tov_eligible_20m


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 5 — Mongolia task monitor (backend-aware)
#
# Queries GEE for every task whose description matches the
# Mongolia naming pattern, regardless of whether the local
# `submitted_tasks_mn` list still holds the references.
# Survives kernel restarts, accidental list wipes, and reruns.
# ============================================================

import ee
from collections import Counter

PATTERN = 'Mongolia_'   # match any task whose description contains this

matching = []
for t in ee.batch.Task.list():
    status = t.status()
    desc   = status.get('description') or ''
    state  = status.get('state') or '?'
    if PATTERN in desc:
        matching.append((desc, state))

if not matching:
    print(f'No tasks found matching "{PATTERN}".')
else:
    matching.sort(key=lambda x: x[0])   # alphabetical by description
    icons = {
        'COMPLETED':        '✅',
        'FAILED':           '❌',
        'RUNNING':          '🔄',
        'READY':            '⏳',
        'CANCELLED':        '🚫',
        'CANCEL_REQUESTED': '🚫',
    }
    print(f'Status for {len(matching)} tasks matching "{PATTERN}":\n')
    for desc, state in matching:
        print(f'  {icons.get(state, "❓")}  {state:<18}  {desc}')

    counts = Counter(state for _, state in matching)
    print('\nSummary:')
    for state, n in sorted(counts.items()):
        print(f'  {state:<18}  {n}')

    active = sum(n for s, n in counts.items() if s in ('READY', 'RUNNING'))
    if active == 0:
        print('\nAll tasks finished. Check Drive for the exported files.')
    else:
        print(f'\n{active} task(s) still active — re-run this cell to refresh.')

Status for 175 tasks matching "Mongolia_":

  ✅  COMPLETED           FailureCount_Mongolia_Abies_sibirica_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Abies_sibirica_29May26_1135_54
  ✅  COMPLETED           FailureCount_Mongolia_Betula_pendula_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Betula_pendula_29May26_1135_54
  ✅  COMPLETED           FailureCount_Mongolia_Betula_platyphylla_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Betula_platyphylla_29May26_1135_54
  ✅  COMPLETED           FailureCount_Mongolia_Caragana_arborescens_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Caragana_arborescens_29May26_1135_54
  ✅  COMPLETED           FailureCount_Mongolia_Caragana_microphylla_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Caragana_microphylla_29May26_1135_54
  ✅  COMPLETED           FailureCount_Mongolia_Elaeagnus_angustifolia_29May26_1126_20
  ✅  COMPLETED           FailureCount_Mongolia_Elaeagnus_a

In [ ]:
# For selecting and downloading multiple files from targeted folder in GDrive where the name of the file ends with the following:

import os, shutil
from google.colab import drive

#drive.mount('/content/drive')   # Already mounted from preivous code

# Source folder to search within
src = '/content/drive/MyDrive/EligibleLandExports'

# New folder to create for content to be downloaded
dst = '/content/blocks_2m_files'

os.makedirs(dst, exist_ok=True)

copied = []
for fname in os.listdir(src):

  # Names ending with this will be coppied

    if fname.endswith('blocks_2m.tif') or fname.endswith('eligible_2m.tif'):
        shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
        copied.append(fname)


print(f'Copied {len(copied)} files:')
for f in copied: print(f'  {f}')

Copied 106 files:
  Assam_Aghunaqa_eligible_2m.tif
  Assam_Badarpur_eligible_2m.tif
  Assam_Agomani_eligible_2m.tif
  Assam_Algapur_eligible_2m.tif
  Assam_Bajengdoba_eligible_2m.tif
  Assam_Amguri_eligible_2m.tif
  Assam_Athibung_eligible_2m.tif
  Assam_Balijan_eligible_2m.tif
  Assam_Barama_eligible_2m.tif
  Assam_Amri_eligible_2m.tif
  Assam_Bajali_eligible_2m.tif
  Assam_Bajiagaon_eligible_2m.tif
  Assam_Banskandi_eligible_2m.tif
  Assam_Barigog_Banbhag_eligible_2m.tif
  Assam_Barjalenga_eligible_2m.tif
  Assam_Balipara_eligible_2m.tif
  Assam_Batabraba_Part_eligible_2m.tif
  Assam_Balijana_eligible_2m.tif
  Assam_Barhampur_eligible_2m.tif
  Assam_Batadrava_eligible_2m.tif
  Assam_Behali_eligible_2m.tif
  Assam_Bechimari_eligible_2m.tif
  Assam_Baska_eligible_2m.tif
  Assam_Betasing_eligible_2m.tif
  Assam_Barpeta_eligible_2m.tif
  Assam_Bezera_eligible_2m.tif
  Assam_Bhandari_eligible_2m.tif
  Assam_Barkhetri_eligible_2m.tif
  Assam_Bhoirymbong_eligible_2m.tif
  Assam_Bezera_Pt_el

In [ ]:
# downloads the selected files identified in previous cell
shutil.make_archive('/content/blocks_2m_files', 'zip', dst)
from google.colab import files
files.download('/content/blocks_2m_files.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>